In [1]:
# =============================================================
# STEP 0: IMPORTS
# =============================================================
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(42)
# active = reproducible | comment out/delete = random

In [2]:
# =============================================================
# STEP 1: GLOBAL PARAMETERS (edit everything here)
# =============================================================

N = 64              # number of subcarriers (IFFT size)
CP_LEN = 16          # cyclic prefix length (samples)
MOD_ORDER = 4        # 4=QPSK, 16=16-QAM, 64=64-QAM

PILOT_SPACING = 4    # every Nth subcarrier is a pilot
PILOT_VALUE = 1 + 1j # known pilot symbol

SNR_DB = 15          # channel SNR for single-run tests
SNR_SWEEP_RANGE = np.arange(0, 21, 2)   # for BER vs SNR plot
NUM_SYMBOLS_PER_SNR = 500                # symbols per SNR point (BER accuracy)

TAU_MAX_SAMPLES = 3              # multipath delay spread (samples)
CHANNEL_TAPS = np.array([1.0, 0.5])   # [direct path, delayed path]

CFO_RANGE = np.linspace(0, 0.15, 8)     # normalized CFO sweep
TIMING_OFFSET_RANGE = np.arange(-4, CP_LEN + 4, 2)  # timing offset sweep (samples)
FIXED_SNR_FOR_IMPAIRMENTS = 20   # high SNR to isolate CFO/timing effects
NUM_SYMBOLS_IMPAIRMENT = 300

# derived (auto-calculated, don't edit)
BITS_PER_SYMBOL = int(np.log2(MOD_ORDER))
pilot_indices = np.arange(0, N, PILOT_SPACING)
data_indices = np.setdiff1d(np.arange(N), pilot_indices)

In [3]:
# =============================================================
# STEP 2: BIT GENERATION
# =============================================================

def generate_bits(num_bits):
    """Generate random 0/1 bits."""
    return np.random.randint(0, 2, num_bits)

num_data_bits = len(data_indices) * BITS_PER_SYMBOL
bits = generate_bits(num_data_bits)

print("Total data bits:", len(bits))
print("First 20 bits:", bits[:20])

Total data bits: 96
First 20 bits: [0 1 0 0 0 1 0 0 0 1 0 0 0 0 1 0 1 1 1 0]


In [4]:
# =============================================================
# STEP 3: QAM/PSK MAPPING
# =============================================================

def qam_mapper(bits, mod_order):
    """Bits -> unit-energy complex QAM/PSK symbols (Gray-coded)."""
    bits_per_sym = int(np.log2(mod_order))
    k = bits_per_sym // 2
    m = int(np.sqrt(mod_order))
    bit_groups = bits.reshape(-1, bits_per_sym)

    def gray_decode(bits_axis):
        val, prev = 0, 0
        for b in bits_axis:
            cur = b ^ prev
            val = (val << 1) | cur
            prev = cur
        return val

    levels = np.arange(m)
    pam_levels = 2 * levels - (m - 1)

    symbols = np.zeros(len(bit_groups), dtype=complex)
    for idx, grp in enumerate(bit_groups):
        i_level = pam_levels[gray_decode(grp[:k])]
        q_level = pam_levels[gray_decode(grp[k:])]
        symbols[idx] = i_level + 1j * q_level

    energy = np.mean(np.abs(symbols) ** 2)
    return symbols / np.sqrt(energy)

def build_constellation_table(mod_order):
    """Lookup table: every bit combo -> its symbol (used by demapper later)."""
    bits_per_sym = int(np.log2(mod_order))
    all_bit_combos = np.array([
        [int(b) for b in format(i, f'0{bits_per_sym}b')]
        for i in range(mod_order)
    ])
    all_symbols = qam_mapper(all_bit_combos.flatten(), mod_order)
    return all_bit_combos, all_symbols

CONSTELLATION_BITS, CONSTELLATION_SYMS = build_constellation_table(MOD_ORDER)

# map data bits to symbols, build full X with pilots inserted
data_symbols = qam_mapper(bits, MOD_ORDER)
X = np.zeros(N, dtype=complex)
X[data_indices] = data_symbols
X[pilot_indices] = PILOT_VALUE

print("X shape:", X.shape)
print("X (first 5):", X[:5])

X shape: (64,)
X (first 5): [ 1.        +1.j         -0.70710678+0.70710678j -0.70710678-0.70710678j
 -0.70710678+0.70710678j  1.        +1.j        ]


In [5]:
# =============================================================
# STEP 4: IFFT (TIME-DOMAIN SIGNAL)
# =============================================================

x = np.fft.ifft(X, n=N)
# frequency domain X -> time domain x (this is the actual OFDM waveform)

print("x shape:", x.shape)
print("x (first 5):", x[:5])

# quick sanity check: FFT should undo IFFT
X_check = np.fft.fft(x, n=N)
print("Max reconstruction error:", np.max(np.abs(X - X_check)))

x shape: (64,)
x (first 5): [0.27209709+0.33838835j 0.04361235+0.1326764j  0.00370775-0.08632414j
 0.03850227-0.10564891j 0.04690752+0.0323739j ]
Max reconstruction error: 4.0029660424867215e-16


In [6]:
# =============================================================
# STEP 5: CYCLIC PREFIX (ADD)
# =============================================================

def add_cyclic_prefix(x, cp_len):
    """Copy last cp_len samples of x, prepend as guard interval."""
    return np.concatenate([x[-cp_len:], x])

x_cp = add_cyclic_prefix(x, CP_LEN)
# x_cp = actual transmitted signal (CP + symbol)

print("x_cp shape:", x_cp.shape)

x_cp shape: (80,)


In [7]:
# =============================================================
# STEP 6: MULTIPATH CHANNEL + AWGN
# =============================================================

def generate_rayleigh_channel(num_subcarriers):
    """One complex Rayleigh gain per subcarrier (freq-domain, flat fading)."""
    h_real = np.random.randn(num_subcarriers) / np.sqrt(2)
    h_imag = np.random.randn(num_subcarriers) / np.sqrt(2)
    return h_real + 1j * h_imag

def apply_multipath(x_cp, taps):
    """Apply time-domain multipath (2-tap channel) via convolution."""
    h_time = np.zeros(TAU_MAX_SAMPLES + 1, dtype=complex)
    h_time[0] = taps[0]
    h_time[TAU_MAX_SAMPLES] = taps[1]
    return np.convolve(x_cp, h_time)[:len(x_cp)]

def add_awgn(signal, snr_db):
    """Add AWGN at given SNR (dB). Returns noisy signal + noise power."""
    snr_linear = 10 ** (snr_db / 10)
    signal_power = np.mean(np.abs(signal) ** 2)
    noise_power = signal_power / snr_linear
    noise = np.sqrt(noise_power / 2) * (
        np.random.randn(len(signal)) + 1j * np.random.randn(len(signal))
    )
    return signal + noise, noise_power

H = generate_rayleigh_channel(N)
x_multipath = apply_multipath(x_cp, CHANNEL_TAPS)
x_rx, noise_power = add_awgn(x_multipath, SNR_DB)

print("H (first 5):", H[:5])
print("x_rx shape:", x_rx.shape)

H (first 5): [-0.93916936-0.24233576j  0.13920191-0.5672957j   0.52217473-0.11404622j
  0.12117567+0.2857071j  -0.08177568+1.33373484j]
x_rx shape: (80,)


In [8]:
# =============================================================
# STEP 7: CYCLIC PREFIX (REMOVE) + FFT
# =============================================================

def remove_cyclic_prefix(x_cp, cp_len):
    """Strip CP samples, leaving N samples for FFT."""
    return x_cp[cp_len:cp_len + N]

x_windowed = remove_cyclic_prefix(x_rx, CP_LEN)
Y = np.fft.fft(x_windowed, n=N)
# Y = received frequency-domain symbols (faded + noisy)

print("Y shape:", Y.shape)
print("Y (first 5):", Y[:5])

Y shape: (64,)
Y (first 5): [ 1.55121515+1.55175598j -1.24802115+0.95353179j -1.13031697-0.82584147j
 -0.73910631+0.9397462j   1.75832894+0.69375911j]


In [9]:
# =============================================================
# STEP 8: PILOT-BASED CHANNEL ESTIMATION
# =============================================================

H_est_at_pilots = Y[pilot_indices] / X[pilot_indices]
# H_hat[k] = Y_pilot[k] / X_pilot[k]

all_idx = np.arange(N)
H_est = (np.interp(all_idx, pilot_indices, H_est_at_pilots.real)
         + 1j * np.interp(all_idx, pilot_indices, H_est_at_pilots.imag))
# interpolate real & imag separately to fill in H for data subcarriers

print("H_est (first 5):", H_est[:5])
print("Mean abs error vs true H:", np.mean(np.abs(H_est - H)))

H_est (first 5): [1.55148556+2.70411907e-04j 1.47012518-1.32868419e-01j
 1.3887648 -2.66007250e-01j 1.30740441-3.99146081e-01j
 1.22604403-5.32284912e-01j]
Mean abs error vs true H: 1.291437985276088


In [10]:
# =============================================================
# STEP 9: EQUALIZATION (ZF / MMSE)
# =============================================================

def zf_equalizer(Y, H_est):
    """Zero-Forcing: X_hat = Y / H_hat."""
    return Y / H_est

def mmse_equalizer(Y, H_est, noise_power, signal_power=1.0):
    """MMSE: noise-aware weighting, more robust in deep fades."""
    noise_to_signal = noise_power / signal_power
    weight = np.conj(H_est) / (np.abs(H_est) ** 2 + noise_to_signal)
    return weight * Y

X_hat_zf = zf_equalizer(Y, H_est)
X_hat_mmse = mmse_equalizer(Y, H_est, noise_power, signal_power=1.0)

print("X_hat_zf (first 5):", X_hat_zf[:5])
print("X_hat_mmse (first 5):", X_hat_mmse[:5])

X_hat_zf (first 5): [ 1.        +1.j         -0.90018896+0.56724769j -0.67522558-0.72399327j
 -0.71785814+0.49962807j  1.        +1.j        ]
X_hat_mmse (first 5): [ 0.9996699 +0.9996699j  -0.8998607 +0.56704084j -0.67495725-0.72370557j
 -0.71755291+0.49941563j  0.99955528+0.99955528j]


In [11]:
# =============================================================
# STEP 10: DEMAPPING (BITS RECOVERY)
# =============================================================

def qam_demapper(rx_symbols, constellation_bits, constellation_syms):
    """Nearest-neighbor demap: closest constellation point -> its bits."""
    bits_out = []
    for sym in rx_symbols:
        distances = np.abs(sym - constellation_syms)
        nearest_idx = np.argmin(distances)
        bits_out.extend(constellation_bits[nearest_idx])
    return np.array(bits_out)

rx_bits_zf = qam_demapper(X_hat_zf[data_indices], CONSTELLATION_BITS, CONSTELLATION_SYMS)
rx_bits_mmse = qam_demapper(X_hat_mmse[data_indices], CONSTELLATION_BITS, CONSTELLATION_SYMS)

print("Bits recovered (ZF, first 20):", rx_bits_zf[:20])
print("Bits recovered (MMSE, first 20):", rx_bits_mmse[:20])

Bits recovered (ZF, first 20): [0 1 0 0 0 1 0 0 0 1 0 0 0 0 1 0 1 1 1 0]
Bits recovered (MMSE, first 20): [0 1 0 0 0 1 0 0 0 1 0 0 0 0 1 0 1 1 1 0]


In [12]:
# =============================================================
# STEP 11: BER CALCULATION
# =============================================================

def compute_ber(tx_bits, rx_bits):
    """Fraction of bits received wrong."""
    errors = np.sum(tx_bits != rx_bits)
    return errors / len(tx_bits), errors

ber_zf, errors_zf = compute_ber(bits, rx_bits_zf)
ber_mmse, errors_mmse = compute_ber(bits, rx_bits_mmse)

print(f"ZF   -> errors: {errors_zf}, BER: {ber_zf:.5f}")
print(f"MMSE -> errors: {errors_mmse}, BER: {ber_mmse:.5f}")

ZF   -> errors: 1, BER: 0.01042
MMSE -> errors: 1, BER: 0.01042


In [13]:
# =============================================================
# STEP 12: PAPR CALCULATION
# =============================================================

def compute_papr(x):
    """PAPR (dB) of a time-domain OFDM symbol."""
    peak_power = np.max(np.abs(x) ** 2)
    avg_power = np.mean(np.abs(x) ** 2)
    return 10 * np.log10(peak_power / avg_power)

papr_db = compute_papr(x)

print(f"PAPR: {papr_db:.2f} dB")

PAPR: 9.85 dB


In [14]:
# =============================================================
# STEP 13: CFO IMPAIRMENT
# =============================================================

def apply_cfo(signal, cfo_normalized, N):
    """Rotate signal by phase growing linearly with n -> simulates CFO."""
    n_idx = np.arange(len(signal))
    return signal * np.exp(1j * 2 * np.pi * cfo_normalized * n_idx / N)

# example: apply a test CFO to x_rx
x_rx_cfo = apply_cfo(x_rx, cfo_normalized=0.05, N=N)

print("x_rx_cfo shape:", x_rx_cfo.shape)

x_rx_cfo shape: (80,)


In [15]:
# =============================================================
# STEP 14: TIMING OFFSET IMPAIRMENT
# =============================================================

def apply_timing_offset(signal, cp_len, N, offset):
    """Shift FFT window start point by `offset` samples (early/late)."""
    start = max(0, min(cp_len + offset, len(signal) - N))
    return signal[start:start + N]

# example: apply a test timing offset to x_rx
x_windowed_offset = apply_timing_offset(x_rx, CP_LEN, N, offset=2)

print("x_windowed_offset shape:", x_windowed_offset.shape)

x_windowed_offset shape: (64,)


In [16]:
# =============================================================
# STEP 15: SNR SWEEP + BER/PAPR PLOTS
# =============================================================

def run_one_symbol(snr_db, equalizer='zf'):
    """Full pipeline for one OFDM symbol -> returns (errors, total_bits, papr_db)."""
    tx_bits = generate_bits(len(data_indices) * BITS_PER_SYMBOL)
    data_syms = qam_mapper(tx_bits, MOD_ORDER)

    X_ = np.zeros(N, dtype=complex)
    X_[data_indices] = data_syms
    X_[pilot_indices] = PILOT_VALUE

    x_ = np.fft.ifft(X_, n=N)
    x_cp_ = add_cyclic_prefix(x_, CP_LEN)

    papr_db_ = compute_papr(x_)

    H_ = generate_rayleigh_channel(N)
    x_mp_ = apply_multipath(x_cp_, CHANNEL_TAPS)
    x_rx_, noise_pow_ = add_awgn(x_mp_, snr_db)

    x_win_ = remove_cyclic_prefix(x_rx_, CP_LEN)
    Y_ = np.fft.fft(x_win_, n=N)

    H_est_pilots_ = Y_[pilot_indices] / X_[pilot_indices]
    H_est_ = (np.interp(all_idx, pilot_indices, H_est_pilots_.real)
              + 1j * np.interp(all_idx, pilot_indices, H_est_pilots_.imag))

    X_hat_ = zf_equalizer(Y_, H_est_) if equalizer == 'zf' \
             else mmse_equalizer(Y_, H_est_, noise_pow_)

    rx_bits_ = qam_demapper(X_hat_[data_indices], CONSTELLATION_BITS, CONSTELLATION_SYMS)
    _, errors_ = compute_ber(tx_bits, rx_bits_)

    return errors_, len(tx_bits), papr_db_


# --- sweep SNR, accumulate BER for ZF and MMSE ---
ber_zf_list, ber_mmse_list, papr_all = [], [], []

for snr in SNR_SWEEP_RANGE:
    e_zf_tot, e_mmse_tot, bits_tot = 0, 0, 0
    for _ in range(NUM_SYMBOLS_PER_SNR):
        e1, t1, papr = run_one_symbol(snr, 'zf')
        e2, t2, _ = run_one_symbol(snr, 'mmse')
        e_zf_tot += e1
        e_mmse_tot += e2
        bits_tot += t1
        papr_all.append(papr)
    ber_zf_list.append(e_zf_tot / bits_tot)
    ber_mmse_list.append(e_mmse_tot / bits_tot)
    print(f"SNR={snr}dB | ZF={ber_zf_list[-1]:.5f} | MMSE={ber_mmse_list[-1]:.5f}")

ber_zf_list = np.array(ber_zf_list)
ber_mmse_list = np.array(ber_mmse_list)
papr_all = np.array(papr_all)


# --- plot BER vs SNR ---
fig = go.Figure()
fig.add_trace(go.Scatter(x=SNR_SWEEP_RANGE, y=ber_zf_list, mode='lines+markers',
                          name='ZF', line=dict(color='crimson')))
fig.add_trace(go.Scatter(x=SNR_SWEEP_RANGE, y=ber_mmse_list, mode='lines+markers',
                          name='MMSE', line=dict(color='seagreen')))
fig.update_layout(title="BER vs SNR", xaxis_title="SNR (dB)", yaxis_title="BER",
                   yaxis_type="log", width=750, height=500)
fig.show()

# --- plot PAPR CCDF ---
thresholds = np.linspace(papr_all.min(), papr_all.max(), 100)
ccdf = [np.mean(papr_all > t) for t in thresholds]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=thresholds, y=ccdf, mode='lines', line=dict(color='darkorange')))
fig2.update_layout(title="PAPR CCDF", xaxis_title="PAPR threshold (dB)",
                    yaxis_title="P(PAPR > threshold)", yaxis_type="log",
                    width=700, height=450)
fig2.show()

SNR=0dB | ZF=0.25625 | MMSE=0.24952
SNR=2dB | ZF=0.19517 | MMSE=0.19446
SNR=4dB | ZF=0.13915 | MMSE=0.13762
SNR=6dB | ZF=0.09044 | MMSE=0.09348
SNR=8dB | ZF=0.05308 | MMSE=0.05306
SNR=10dB | ZF=0.02835 | MMSE=0.02904
SNR=12dB | ZF=0.01346 | MMSE=0.01325
SNR=14dB | ZF=0.00479 | MMSE=0.00494
SNR=16dB | ZF=0.00108 | MMSE=0.00102
SNR=18dB | ZF=0.00017 | MMSE=0.00017
SNR=20dB | ZF=0.00000 | MMSE=0.00000


In [17]:
# =============================================================
# STEP 16: CFO/TIMING OFFSET SWEEP + PLOTS
# =============================================================

def run_one_symbol_impaired(snr_db, cfo_normalized=0.0, timing_offset=0):
    """Full pipeline with optional CFO + timing offset -> (errors, total_bits)."""
    tx_bits = generate_bits(len(data_indices) * BITS_PER_SYMBOL)
    data_syms = qam_mapper(tx_bits, MOD_ORDER)

    X_ = np.zeros(N, dtype=complex)
    X_[data_indices] = data_syms
    X_[pilot_indices] = PILOT_VALUE

    x_ = np.fft.ifft(X_, n=N)
    x_cp_ = add_cyclic_prefix(x_, CP_LEN)
    x_mp_ = apply_multipath(x_cp_, CHANNEL_TAPS)
    x_rx_, noise_pow_ = add_awgn(x_mp_, snr_db)

    x_rx_ = apply_cfo(x_rx_, cfo_normalized, N)
    x_win_ = apply_timing_offset(x_rx_, CP_LEN, N, timing_offset)

    Y_ = np.fft.fft(x_win_, n=N)

    H_est_pilots_ = Y_[pilot_indices] / X_[pilot_indices]
    H_est_ = (np.interp(all_idx, pilot_indices, H_est_pilots_.real)
              + 1j * np.interp(all_idx, pilot_indices, H_est_pilots_.imag))

    X_hat_ = mmse_equalizer(Y_, H_est_, noise_pow_)
    rx_bits_ = qam_demapper(X_hat_[data_indices], CONSTELLATION_BITS, CONSTELLATION_SYMS)
    _, errors_ = compute_ber(tx_bits, rx_bits_)

    return errors_, len(tx_bits)


# --- sweep CFO (fixed high SNR to isolate ICI effect) ---
ber_vs_cfo = []
for eps in CFO_RANGE:
    e_tot, t_tot = 0, 0
    for _ in range(NUM_SYMBOLS_IMPAIRMENT):
        e, t = run_one_symbol_impaired(FIXED_SNR_FOR_IMPAIRMENTS, cfo_normalized=eps)
        e_tot += e
        t_tot += t
    ber_vs_cfo.append(e_tot / t_tot)
    print(f"CFO={eps:.3f} | BER={ber_vs_cfo[-1]:.5f}")
ber_vs_cfo = np.array(ber_vs_cfo)

# --- sweep timing offset ---
ber_vs_timing = []
for off in TIMING_OFFSET_RANGE:
    e_tot, t_tot = 0, 0
    for _ in range(NUM_SYMBOLS_IMPAIRMENT):
        e, t = run_one_symbol_impaired(FIXED_SNR_FOR_IMPAIRMENTS, timing_offset=off)
        e_tot += e
        t_tot += t
    ber_vs_timing.append(e_tot / t_tot)
    print(f"Offset={off} | BER={ber_vs_timing[-1]:.5f}")
ber_vs_timing = np.array(ber_vs_timing)


# --- plot BER vs CFO ---
fig = go.Figure()
fig.add_trace(go.Scatter(x=CFO_RANGE, y=ber_vs_cfo, mode='lines+markers',
                          line=dict(color='crimson')))
fig.update_layout(title=f"BER vs Normalized CFO (SNR={FIXED_SNR_FOR_IMPAIRMENTS}dB)",
                   xaxis_title="Normalized CFO", yaxis_title="BER",
                   yaxis_type="log", width=700, height=450)
fig.show()

# --- plot BER vs timing offset (with CP safe zone shaded) ---
safe_margin = CP_LEN - TAU_MAX_SAMPLES
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=TIMING_OFFSET_RANGE, y=ber_vs_timing, mode='lines+markers',
                           line=dict(color='seagreen')))
fig2.add_vrect(x0=0, x1=safe_margin, fillcolor="lightgreen", opacity=0.2, line_width=0,
               annotation_text="CP safe zone", annotation_position="top left")
fig2.update_layout(title=f"BER vs Timing Offset (SNR={FIXED_SNR_FOR_IMPAIRMENTS}dB)",
                    xaxis_title="Timing offset (samples)", yaxis_title="BER",
                    yaxis_type="log", width=750, height=450)
fig2.show()

CFO=0.000 | BER=0.00000
CFO=0.021 | BER=0.00003
CFO=0.043 | BER=0.00003
CFO=0.064 | BER=0.00007
CFO=0.086 | BER=0.00056
CFO=0.107 | BER=0.00215
CFO=0.129 | BER=0.00483
CFO=0.150 | BER=0.00931
Offset=-4 | BER=0.02090
Offset=-2 | BER=0.00885
Offset=0 | BER=0.00000
Offset=2 | BER=0.00000
Offset=4 | BER=0.00000
Offset=6 | BER=0.00000
Offset=8 | BER=0.00000
Offset=10 | BER=0.00000
Offset=12 | BER=0.00000
Offset=14 | BER=0.00000
Offset=16 | BER=0.00000
Offset=18 | BER=0.00000
